In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import random
from torch.utils.data import random_split
import os

class Config:
    batch_size = 128
    epochs = 150
    lr = 1e-3
    lambdas = [1e-6, 1e-5, 1e-4]
    sparsity_threshold = 1e-2
    patience = 3

cfg = Config()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

os.makedirs("results", exist_ok=True)

In [2]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform)

train_size = int(0.9 * len(full_train))
val_size = len(full_train) - train_size

trainset, valset = random_split(full_train, [train_size, val_size])

trainloader = torch.utils.data.DataLoader(trainset, batch_size=cfg.batch_size, shuffle=True)
valloader = torch.utils.data.DataLoader(valset, batch_size=cfg.batch_size)
testloader = torch.utils.data.DataLoader(testset, batch_size=cfg.batch_size)

100%|██████████| 170M/170M [00:12<00:00, 14.0MB/s]


In [3]:
class PrunableLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_f))
        self.gate_scores = nn.Parameter(torch.randn(out_f, in_f))

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)
        return F.linear(x, self.weight * gates, self.bias)

    def gates(self):
        return torch.sigmoid(self.gate_scores)

In [4]:
class PrunableMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = PrunableLinear(3*32*32, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

    def all_gates(self):
        return torch.cat([
            self.fc1.gates().view(-1),
            self.fc2.gates().view(-1),
            self.fc3.gates().view(-1)
        ])

class BaselineMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3*32*32, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [5]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

def evaluate_loss(model, loader):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total += loss_fn(model(x), y).item()
    return total / len(loader)

def sparsity(model):
    g = model.all_gates().detach().cpu().numpy()
    return 100 * np.mean(g < cfg.sparsity_threshold)

In [6]:
class EarlyStop:
    def __init__(self):
        self.best = float('inf')
        self.count = 0
        self.best_w = None

    def step(self, val_loss, model):
        if val_loss < self.best:
            self.best = val_loss
            self.count = 0
            self.best_w = model.state_dict()
            return False
        else:
            self.count += 1
            return self.count >= cfg.patience

In [7]:
def train_baseline():
    model = BaselineMLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(cfg.epochs):
        model.train()
        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()

    return model

def train_prunable(lam):
    model = PrunableMLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-6)
    loss_fn = nn.CrossEntropyLoss()
    stopper = EarlyStop()

    metrics = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "sp":[], "lr":[]}

    for ep in range(cfg.epochs):
        model.train()
        tl, correct, total = 0,0,0

        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()

            out = model(x)
            loss = loss_fn(out, y) + lam * torch.sum(model.all_gates())
            loss.backward()
            opt.step()

            tl += loss.item()
            correct += (out.argmax(1)==y).sum().item()
            total += y.size(0)

        train_acc = 100*correct/total
        val_loss = evaluate_loss(model, valloader)
        val_acc = evaluate(model, valloader)
        sp = sparsity(model)

        scheduler.step()
        lr = opt.param_groups[0]['lr']

        metrics["train_loss"].append(tl/len(trainloader))
        metrics["val_loss"].append(val_loss)
        metrics["train_acc"].append(train_acc)
        metrics["val_acc"].append(val_acc)
        metrics["sp"].append(sp)
        metrics["lr"].append(lr)

        print(f"Ep{ep+1} | LR {lr:.6f} | ValAcc {val_acc:.2f} | Sp {sp:.2f}")

        if stopper.step(val_loss, model):
            print("Early stop")
            break

    model.load_state_dict(stopper.best_w)
    return model, metrics

In [8]:
baseline = train_baseline()
base_acc = evaluate(baseline, testloader)

results = []

for lam in cfg.lambdas:
    model, m = train_prunable(lam)
    acc = evaluate(model, testloader)
    sp = sparsity(model)
    results.append((lam, acc, sp, m, model))

Ep1 | LR 0.001000 | ValAcc 37.52 | Sp 0.00
Ep2 | LR 0.001000 | ValAcc 42.18 | Sp 0.00
Ep3 | LR 0.000999 | ValAcc 43.46 | Sp 0.00
Ep4 | LR 0.000998 | ValAcc 46.54 | Sp 0.00
Ep5 | LR 0.000997 | ValAcc 47.76 | Sp 0.00
Ep6 | LR 0.000996 | ValAcc 48.94 | Sp 0.00
Ep7 | LR 0.000995 | ValAcc 49.04 | Sp 0.00
Ep8 | LR 0.000993 | ValAcc 49.98 | Sp 0.01
Ep9 | LR 0.000991 | ValAcc 49.96 | Sp 0.01
Ep10 | LR 0.000989 | ValAcc 51.16 | Sp 0.01
Ep11 | LR 0.000987 | ValAcc 51.16 | Sp 0.02
Ep12 | LR 0.000984 | ValAcc 51.56 | Sp 0.02
Ep13 | LR 0.000982 | ValAcc 52.28 | Sp 0.04
Ep14 | LR 0.000979 | ValAcc 52.34 | Sp 0.05
Ep15 | LR 0.000976 | ValAcc 53.00 | Sp 0.07
Ep16 | LR 0.000972 | ValAcc 52.76 | Sp 0.09
Ep17 | LR 0.000969 | ValAcc 53.72 | Sp 0.13
Ep18 | LR 0.000965 | ValAcc 53.94 | Sp 0.17
Ep19 | LR 0.000961 | ValAcc 54.26 | Sp 0.22
Ep20 | LR 0.000957 | ValAcc 54.04 | Sp 0.29
Early stop
Ep1 | LR 0.001000 | ValAcc 37.32 | Sp 0.00
Ep2 | LR 0.001000 | ValAcc 41.22 | Sp 0.00
Ep3 | LR 0.000999 | ValAcc 44.18

In [9]:
def plot_metrics(m, lam):
    e = range(1,len(m["train_loss"])+1)

    plt.figure(); plt.plot(e,m["train_loss"]); plt.plot(e,m["val_loss"]); plt.title("Loss"); plt.savefig(f"results/loss_{lam}.png"); plt.close()
    plt.figure(); plt.plot(e,m["train_acc"]); plt.plot(e,m["val_acc"]); plt.title("Accuracy"); plt.savefig(f"results/acc_{lam}.png"); plt.close()
    plt.figure(); plt.plot(e,m["sp"]); plt.title("Sparsity"); plt.savefig(f"results/sp_{lam}.png"); plt.close()
    plt.figure(); plt.plot(e,m["lr"]); plt.title("LR"); plt.savefig(f"results/lr_{lam}.png"); plt.close()

for lam,_,_,m,_ in results:
    plot_metrics(m, lam)

plt.figure()
plt.plot([r[2] for r in results],[r[1] for r in results],'o-')
plt.xlabel("Sparsity"); plt.ylabel("Accuracy")
plt.savefig("results/tradeoff.png"); plt.close()

best = max(results, key=lambda x: x[1])[4]
g = best.all_gates().detach().cpu().numpy()

plt.figure()
plt.hist(g, bins=50)
plt.savefig("results/gates.png")
plt.close()

In [10]:
print("\nFinal:")
print(f"Baseline Acc: {base_acc:.2f}%")

for lam, acc, sp,_,_ in results:
    print(f"λ={lam} | Acc={acc:.2f}% | Sparsity={sp:.2f}%")


Final:
Baseline Acc: 58.19%
λ=1e-06 | Acc=53.93% | Sparsity=0.29%
λ=1e-05 | Acc=57.24% | Sparsity=39.54%
λ=0.0001 | Acc=51.14% | Sparsity=15.79%
